# 04｜JAX 执行模型：PRNG、vmap、scan 与 JIT

对应[第 04 章](../course/04-jax-mjx-warp-and-brax.md)。目标不是记 API，而是能预测 shape、随机行为、编译边界和计时陷阱。

## 学习目标

理解显式 key；把单环境函数提升为 batch；用 scan 表达时间递推；区分 trace/compile、异步 dispatch 与真实执行；将这些概念映射到 MJX/Warp/Brax。

## 本节知识地图

本节只学 6 个知识点。先判断每个概念改变的是“随机状态、环境维、时间维还是执行方式”。

| 知识点 | 一句话解释 | 在 Panda 中对应什么 | 掌握检查 |
| --- | --- | --- | --- |
| 纯函数 | 输出只由显式输入决定，不偷偷改全局状态 | env reset/step | 能画出 state 输入输出 |
| PRNG key | 随机状态是显式数组，消费前必须 split | reset 随机化 | 不复用 sample key |
| `vmap` | 给单样本函数增加环境 batch 维 | 许多并行世界 | 推导 env 维 shape |
| `scan` | 沿时间反复应用同一递推函数 | rollout | 推导 time 维 shape |
| `jit` | 按 shape/dtype/静态结构编译数组程序 | 首轮等待与重编译 | 正确进行同步计时 |
| 栈边界 | MJX/Warp 做设备物理，Brax 组织 PPO | 训练全链路 | 能从 XML 追到 update |

## 关键概念与符号

| 名词 | 改变什么 | 不要混淆 |
| --- | --- | --- |
| batch | 同一时刻的多个环境 | 不是时间步 |
| carry | `scan` 传到下一步的状态 | 不自动带 time 维 |
| collected output | 每步需要保存的结果 | 会堆出 time 维 |
| trace/compile | 根据抽象 shape 建程序并生成设备代码 | 不是最终数值执行 |
| dispatch | 把工作提交给设备 | 可能在计时结束后仍执行 |

> **Shape 口诀：** 原函数给 feature，`vmap` 加 env，`scan` 加 time；典型轨迹为 `[time, env, feature]`。

## 先预测

1. 同一个 key 调两次 `uniform` 会得到相同还是不同数组？
2. 单样本 `[3] -> scalar` 经 `vmap` 接收 `[8,3]` 后输出 shape 是什么？
3. `scan` 10 步、32 个环境、单 observation `[8]`，收集结果的概念 shape 是什么？
4. shape 从 `(8,3)` 改成 `(16,3)` 是否保证复用原 JIT 编译？

## 运行与观察

先确认 JAX backend。Mac 显示 CPU 是正确结果；这不影响概念实验，但不能证明正式视觉 PPO 吞吐。

In [ ]:
from pathlib import Path
import sys
import time

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from course_utils import assert_course_kernel

assert_course_kernel(ROOT)
print('JAX:', jax.__version__)
print('Backend:', jax.default_backend())
print('Devices:', jax.devices())

### 1. Key 是显式输入，不是可变全局状态

复用 key 会精确复现同一随机样本。正确模式是保留一个 future key，只消费一次 sample key。

In [ ]:
key = jax.random.key(7)
repeated_a = jax.random.uniform(key, (3,))
repeated_b = jax.random.uniform(key, (3,))
future_key, sample_key = jax.random.split(key)
split_sample = jax.random.uniform(sample_key, (3,))
print('same key equal: ', bool(jnp.array_equal(repeated_a, repeated_b)))
print('split key differs:', not bool(jnp.array_equal(repeated_a, split_sample)))
assert bool(jnp.array_equal(repeated_a, repeated_b))
assert not bool(jnp.array_equal(repeated_a, split_sample))

### 2. vmap 增加环境 batch 维

先写只处理一行的纯函数，再让 JAX 变换它。数值应与 Python 显式 map 一致，但执行表达不同。

In [ ]:
def energy(vector):
    return jnp.sum(vector * vector)

batch = jnp.arange(24, dtype=jnp.float32).reshape(8, 3)
vectorized_energy = jax.vmap(energy)
energies = vectorized_energy(batch)
expected = jnp.stack([energy(row) for row in batch])
print('input shape:', batch.shape, 'output shape:', energies.shape)
assert energies.shape == (8,)
assert bool(jnp.allclose(energies, expected))

### 3. scan 增加时间维

carry 是当前 position；每一步 action 产生新 carry 和需要收集的输出。最终 carry 没有 time 维，收集的 trajectory 有。

In [ ]:
def scan_step(position, action):
    next_position = position + action
    return next_position, next_position

initial_positions = jnp.zeros((4, 2))  # 4 个并行世界
actions = jnp.ones((3, 4, 2))          # 3 个时间步
final_positions, trajectory = jax.lax.scan(scan_step, initial_positions, actions)
print('trajectory:', trajectory.shape, 'final carry:', final_positions.shape)
assert trajectory.shape == (3, 4, 2)
assert final_positions.shape == (4, 2)
assert bool(jnp.allclose(trajectory[-1], 3.0))

### 4. JIT 冷调用与缓存调用

JAX 异步提交工作，所以停止计时前必须 `block_until_ready()`。不要断言第一次必然快或慢多少；共享机器和缓存会影响绝对值。

In [ ]:
compiled_energy = jax.jit(vectorized_energy)

def timed(function, value):
    started = time.perf_counter()
    result = function(value)
    jax.block_until_ready(result)
    return result, time.perf_counter() - started

compiled_result, cold_seconds = timed(compiled_energy, batch)
_, cached_seconds = timed(compiled_energy, batch)
print(f'cold={cold_seconds:.6f}s cached={cached_seconds:.6f}s')
assert bool(jnp.allclose(compiled_result, energies))

plt.figure(figsize=(5, 3))
plt.bar(['first call', 'same-shape call'], [cold_seconds, cached_seconds], color=['tab:orange', 'tab:blue'])
plt.ylabel('wall seconds (synchronized)'); plt.title('JIT timing on this machine')
plt.show()

## 动手修改

把 `new_batch_size` 改为 8、16、32。每次先预测输出 shape，并观察换 shape 后调用耗时。不要把一次墙钟值当成可靠 benchmark；这里只验证“shape 是编译接口的一部分”。最后恢复 16。

In [ ]:
new_batch_size = 16
new_batch = jnp.arange(new_batch_size * 3, dtype=jnp.float32).reshape(new_batch_size, 3)
new_result, new_shape_seconds = timed(compiled_energy, new_batch)
print('new input:', new_batch.shape, 'new output:', new_result.shape, f'time={new_shape_seconds:.6f}s')

## 自测

运行前回答：`[time, env, feature]` 三维各来自 scan、vmap 还是原函数？

In [ ]:
assert new_batch_size == 16
assert new_result.shape == (16,)
assert trajectory.shape == (3, 4, 2)
assert cold_seconds >= 0 and cached_seconds >= 0 and new_shape_seconds >= 0
assert jax.default_backend() in {'cpu', 'gpu', 'tpu'}
print('PASS: PRNG, vmap batch, scan time, synchronized JIT timing, and shape change')

## 学完请记住

关闭本页后，你应能脱稿说出：

1. JAX 的随机性由显式 key 控制，复用 key 会复现样本；
2. `vmap` 增加 env 维，`scan` 增加 time 维；
3. `scan` 的 final carry 与 collected trajectory 具有不同 shape；
4. `jit` 冷调用包含编译，换 shape 可能重编译；
5. GPU 运算可能异步，计时前要 `block_until_ready()`；
6. reset/eval 通过不能证明 gradient update 不会 OOM。

若 shape 说不清，先标出原函数 feature，再依次添加 env 和 time 维。

## 反思与记录

在 `notes/04-notebook-reflection.md` 画出：MJCF → MuJoCo model → MJX/Warp batched state → RGB → CNN → Brax PPO。再回答：

1. 为什么同 seed 加倍 `num_envs` 不保证前半环境随机轨迹不变？
2. 为什么 reset/eval 能运行不代表第一轮 gradient update 不会 OOM？
3. 当前 Mac 实验验证了什么，没有验证什么？

最后独立完成 [`04_jax_transforms_exercise.py`](../labs/starter/04_jax_transforms_exercise.py)。